<a href="https://colab.research.google.com/github/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised/blob/main/erps_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trial-level ERP analysis of sentence-final expectancy and semantic integration

This notebook runs the ERP part of the Toffolo et al. (2022) N400Stimset reanalysis. The original study tested ERP responses to sentence-final congruent and incongruent words, with effects reported for the Recognition Potential (RP), N400, and Late Positive Component (LPC/P600). The present analysis keeps the sentence-final ERP focus and uses the predictor table created in the stimulus notebook to model retained ERP/EEG trials at the trial level.

This notebook belongs to the GitHub repository for the Toffolo et al. language reanalysis:  
[https://github.com/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised](https://github.com/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised)

The stimulus notebook created the GLM-ready table of item-level predictors. That table keeps `stim_key` as the stimulus identifier and contains predictors for each sentence-final target word, including human cloze probability, GPT-2 surprisal, LLM-derived expectancy, semantic similarity, lexical frequency, word length, phonology, syntactic complexity, and affective features.

This ERP notebook uses the processed N400Stimset ERP derivatives and epoched EEG files: subject-level ERP `.mat` files, paired `*_trialrej.tsv` files, subject event information, and EEGLAB epoch files for the CP, GA, LD, Order, and Time analysis schemes. These files contain the retained trials after preprocessing and rejection.

The notebook runs three scripts. `erp_analysis/export_erp_long.py` creates `ALL_subjects_ALL_erp_trial_lookup.tsv`, a retained-trial lookup that records the subject, analysis scheme, condition, retained-trial number, EEGLAB event information, original event row, stimulus file, and `stim_key` for every retained ERP trial. `eeg_analysis/prepare_limo_design_matrix.py` uses that lookup to create subject-specific GLM design matrices by attaching the stimulus predictors through `stim_key`. `eeg_analysis/run_limo_first_level_glm.py` pairs each subject’s epoched EEG file with the matching design matrix and estimates beta and t values for the selected predictors across channels and timepoints.

The outputs of this notebook are the retained-trial lookup, the subject-specific GLM design matrices, and the first-level EEG/ERP GLM results. These outputs test whether sentence-final expectancy, surprisal, semantic fit, lexical properties, phonological structure, syntactic complexity, and affective tone explain variation in RP, N400/PN400, and LPC/P600 activity.

In [1]:
%cd /content

!rm -rf Semantically_Incongruent_or_Congruent_Eggplants_revised
!git clone https://github.com/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised.git

%cd /content/Semantically_Incongruent_or_Congruent_Eggplants_revised

/content
Cloning into 'Semantically_Incongruent_or_Congruent_Eggplants_revised'...
remote: Enumerating objects: 418, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 418 (delta 92), reused 51 (delta 31), pack-reused 280 (from 1)
Receiving objects: 100% (418/418), 94.80 MiB | 17.76 MiB/s, done.
Resolving deltas: 100% (240/240), done.
/content/Semantically_Incongruent_or_Congruent_Eggplants_revised


In [2]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 78.7 MB/s eta 0:00:00


In [3]:
!unzip language_outputs.zip

Archive:  language_outputs.zip
   creating: language_outputs/
  inflating: language_outputs/ALL_language_metrics.tsv  
  inflating: language_outputs/ALL_predictor_diagnostics_vif.tsv  
  inflating: language_outputs/ALL_predictor_diagnostics_correlations.tsv  
  inflating: language_outputs/ALL_language_metrics_GLM.tsv  


**Note**: Due to their size, the N400 ERP derivatives are stored on Google Drive and mounted during notebook execution.

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!unzip -o erps.zip

Archive:  erps.zip
  inflating: erps/task-N400Stimset_erp-CP_trialrej.json  
  inflating: erps/task-N400Stimset_erp-GA_filter.json  
  inflating: erps/task-N400Stimset_erp-GA_trialrej.json  
  inflating: erps/task-N400Stimset_erp-LD_trialrej.json  
  inflating: erps/task-N400Stimset_erp-Order_trialrej.json  
  inflating: erps/task-N400Stimset_erp-Time_trialrej.json  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-CP.mat  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-CP_trialrej.tsv  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-GA.mat  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-GA_trialrej.tsv  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-LD.mat  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-LD_trialrej.tsv  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-Order.mat  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-Order_trialrej.tsv  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-Time.mat  
  inflating: erps/sub-01/sub

In [ ]:
%cd /content/Semantically_Incongruent_or_Congruent_Eggplants_revised/

/content/Semantically_Incongruent_or_Congruent_Eggplants_revised


# Export retained ERP trials

This section creates the retained-trial lookup used to connect the ERP data with the stimulus predictors.

The export script reads the ERP `.mat` files, the matching `*_trialrej.tsv` files, the subject event information, and the stimulus lookup. It runs across the CP, GA, LD, Order, and Time analysis schemes.

The output is:

`ALL_subjects_ALL_erp_trial_lookup.tsv`

For each retained ERP trial, the table stores the subject, analysis scheme, condition, retained-trial number, event information, stimulus file, and `stim_key`.

# Build subject-specific GLM design matrices

This section creates the design matrices used in the first-level ERP/EEG GLM.

The script combines the retained-trial lookup with the GLM-ready stimulus predictor table using `stim_key`. Each output design matrix contains the retained trials for one subject and one ERP analysis scheme, with the corresponding language predictors attached to each epoch.

# Run first-level EEG/ERP GLMs

This section fits the subject-level EEG/ERP GLM.

Each subject’s epoched EEG file is paired with the matching design matrix. The model estimates beta and t values for the selected language predictors across channels and timepoints.